# TF-IDF Cosine Similarity on the 20 Newsgroups Dataset

## Objective

This experiment uses the **20 Newsgroups dataset** to investigate whether TF-IDF can identify documents belonging to the same class using cosine similarity, and whether adding bigrams improves this ability.

### Experimental setup

- **Dataset:** 20 Newsgroups
- **Categories:** `comp.graphics`, `rec.sport.hockey`, `sci.space`
- **Documents:** 100 randomly selected from each category
- **Total documents:** 300
- **Random seed:** 42
- **Preprocessing:** lowercasing, tokenization, stop-word removal, lemmatization
- **Representations:** TF-IDF unigrams; TF-IDF unigrams + bigrams
- **Similarity:** cosine similarity implemented from scratch using NumPy

The same 300 documents are used for both representations so that the comparison is fair.

## 1. Import Libraries and Set Experimental Parameters

In [1]:
import random
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

RANDOM_STATE = 42
DOCUMENTS_PER_CLASS = 100

CATEGORIES = [
    "comp.graphics",
    "rec.sport.hockey",
    "sci.space"
]

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Categories:", CATEGORIES)
print("Documents per category:", DOCUMENTS_PER_CLASS)
print("Random seed:", RANDOM_STATE)

Categories: ['comp.graphics', 'rec.sport.hockey', 'sci.space']
Documents per category: 100
Random seed: 42


## 2. Load the 20 Newsgroups Dataset

Headers, footers, and quoted replies are removed so that the experiment focuses on the textual content of the documents rather than email metadata.

In [2]:
dataset = fetch_20newsgroups(
    subset="all",
    categories=CATEGORIES,
    shuffle=True,
    random_state=RANDOM_STATE,
    remove=("headers", "footers", "quotes")
)

print("Available categories:")
print(dataset.target_names)

print("\nNumber of available documents:")
print(len(dataset.data))

Available categories:
['comp.graphics', 'rec.sport.hockey', 'sci.space']

Number of available documents:
2959


## 3. Randomly Select 100 Documents from Each Category

Exactly 100 documents are selected without replacement from each category.

A fixed random seed is used so that the experiment can be reproduced.

In [3]:
rng = np.random.default_rng(RANDOM_STATE)

documents = []
labels = []

for category_id, category_name in enumerate(dataset.target_names):
    category_indices = np.where(dataset.target == category_id)[0]

    selected_indices = rng.choice(
        category_indices,
        size=DOCUMENTS_PER_CLASS,
        replace=False
    )

    for index in selected_indices:
        documents.append(dataset.data[index])
        labels.append(category_name)

documents = np.array(documents, dtype=object)
labels = np.array(labels)

print("Total selected documents:", len(documents))
print("\nDocuments per category:")
print(pd.Series(labels).value_counts())

Total selected documents: 300

Documents per category:
comp.graphics       100
rec.sport.hockey    100
sci.space           100
Name: count, dtype: int64


## 4. Preprocess the Documents

The required preprocessing operations are performed in the following order:

1. Lowercasing
2. Tokenization
3. Stop-word removal
4. Lemmatization

In [4]:
import nltk

nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Lowercasing
    text = text.lower()

    # Tokenization
    tokens = word_tokenize(text)

    processed_tokens = []

    for token in tokens:
        # Keep alphabetic words
        if not token.isalpha():
            continue

        # Stop-word removal
        if token in stop_words:
            continue

        # Lemmatization
        token = lemmatizer.lemmatize(token)
        processed_tokens.append(token)

    return " ".join(processed_tokens)

processed_documents = [
    preprocess(document)
    for document in documents
]

print("Preprocessing completed.")

Preprocessing completed.


In [5]:
# Example of the preprocessing result

example_index = 0

print("Class:", labels[example_index])

print("\nOriginal document:")
print(documents[example_index][:1000])

print("\nProcessed document:")
print(processed_documents[example_index][:1000])

Class: comp.graphics

Original document:



    ^^^
If french is your language, try counting in french in stead, maybe
it will work better.... :-)

Processed document:
french language try counting french stead maybe work better


## 5. TF-IDF Representation

Two TF-IDF representations are created:

### A. Unigrams

Each individual word is a feature.

### B. Unigrams + Bigrams

Both individual words and adjacent two-word sequences are features.

The same documents and preprocessing are used for both representations.

In [6]:
# TF-IDF with unigrams

unigram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b"
)

X_unigram = unigram_vectorizer.fit_transform(
    processed_documents
)

# TF-IDF with unigrams and bigrams

bigram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b"
)

X_bigram = bigram_vectorizer.fit_transform(
    processed_documents
)

print("Unigram TF-IDF shape:", X_unigram.shape)
print("Unigram + bigram TF-IDF shape:", X_bigram.shape)

Unigram TF-IDF shape: (300, 6598)
Unigram + bigram TF-IDF shape: (300, 31528)


## 6. Manual Cosine Similarity Using NumPy

For two TF-IDF vectors $A$ and $B$:

$$
\cos(A,B)=\frac{A\cdot B}{\|A\|\|B\|}
$$

No pre-built cosine similarity function is used.

In [7]:
def cosine_similarity_numpy(vector_a, vector_b):
    """Cosine similarity implemented from scratch using NumPy."""

    vector_a = np.asarray(vector_a).ravel()
    vector_b = np.asarray(vector_b).ravel()

    dot_product = np.dot(vector_a, vector_b)

    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

## 7. Calculate the Cosine Similarity of Two Documents Manually

Documents 0 and 1 are selected.

The calculation below explicitly displays:

- the dot product
- the norm of document 0
- the norm of document 1
- the numerator
- the denominator
- the final cosine similarity

This demonstrates the cosine similarity calculation directly from the TF-IDF vectors.

In [8]:
document_a = 0
document_b = 1

vector_a = X_unigram[document_a].toarray().ravel()
vector_b = X_unigram[document_b].toarray().ravel()

dot_product = np.dot(vector_a, vector_b)
norm_a = np.linalg.norm(vector_a)
norm_b = np.linalg.norm(vector_b)
denominator = norm_a * norm_b

manual_cosine = cosine_similarity_numpy(
    vector_a,
    vector_b
)

print("Document A:", document_a)
print("Class A:", labels[document_a])

print("\nDocument B:", document_b)
print("Class B:", labels[document_b])

print("\nDot product:")
print(f"{dot_product:.10f}")

print("\nNorm of Document A:")
print(f"{norm_a:.10f}")

print("\nNorm of Document B:")
print(f"{norm_b:.10f}")

print("\nDenominator:")
print(f"{denominator:.10f}")

print("\nCosine similarity:")
print(f"{manual_cosine:.10f}")

Document A: 0
Class A: comp.graphics

Document B: 1
Class B: comp.graphics

Dot product:
0.0000000000

Norm of Document A:
1.0000000000

Norm of Document B:
1.0000000000

Denominator:
1.0000000000

Cosine similarity:
0.0000000000


## 8. Pairwise Cosine Similarity from Scratch Using NumPy

The following implementation calculates cosine similarity between every pair of documents.

It does **not** use `sklearn.metrics.pairwise.cosine_similarity` or any other pre-built cosine similarity function.

In [9]:
def cosine_similarity_matrix(X):
    """
    Compute the complete pairwise cosine similarity matrix
    using NumPy operations.
    """

    X_dense = X.toarray()

    # Pairwise dot products
    dot_products = np.dot(X_dense, X_dense.T)

    # L2 norm for every document
    norms = np.linalg.norm(X_dense, axis=1)

    # Pairwise products of norms
    denominator = np.outer(norms, norms)

    # Prevent division by zero
    denominator[denominator == 0] = 1.0

    return dot_products / denominator

In [10]:
unigram_similarity = cosine_similarity_matrix(X_unigram)
bigram_similarity = cosine_similarity_matrix(X_bigram)

print("Unigram similarity matrix shape:",
      unigram_similarity.shape)

print("Unigram + bigram similarity matrix shape:",
      bigram_similarity.shape)

Unigram similarity matrix shape: (300, 300)
Unigram + bigram similarity matrix shape: (300, 300)


## 9. Find the 5 Most Similar Documents for Every Document

For every document, the document itself is excluded. The five remaining documents with the highest cosine similarity are then selected.

The corresponding document classes are reported.

In [11]:
def find_top_5(similarity_matrix, labels):
    rows = []

    n_documents = similarity_matrix.shape[0]

    for i in range(n_documents):
        scores = similarity_matrix[i].copy()

        # Do not retrieve the document itself
        scores[i] = -1.0

        top_indices = np.argsort(scores)[-5:][::-1]

        for rank, j in enumerate(top_indices, start=1):
            rows.append({
                "Document": i,
                "Rank": rank,
                "Similar Document": j,
                "Cosine Similarity": scores[j],
                "Document Class": labels[i],
                "Similar Document Class": labels[j]
            })

    return pd.DataFrame(rows)

unigram_top5 = find_top_5(
    unigram_similarity,
    labels
)

bigram_top5 = find_top_5(
    bigram_similarity,
    labels
)

### Top-5 Similar Documents — TF-IDF Unigrams

In [12]:
display(
    unigram_top5
)

,Document,Rank,Similar Document,Cosine Similarity,Document Class,Similar Document Class
0,0,1,54,0.151596,comp.graphics,comp.graphics
1,0,2,118,0.093399,comp.graphics,rec.sport.hockey
2,0,3,208,0.084501,comp.graphics,sci.space
3,0,4,205,0.077589,comp.graphics,sci.space
4,0,5,79,0.068512,comp.graphics,comp.graphics
...,...,...,...,...,...,...
1495,299,1,191,0.072664,sci.space,rec.sport.hockey
1496,299,2,24,0.064560,sci.space,comp.graphics
1497,299,3,26,0.060348,sci.space,comp.graphics
1498,299,4,282,0.053169,sci.space,sci.space


### Top-5 Similar Documents — TF-IDF Unigrams + Bigrams

In [13]:
display(
    bigram_top5
)

,Document,Rank,Similar Document,Cosine Similarity,Document Class,Similar Document Class
0,0,1,54,0.068935,comp.graphics,comp.graphics
1,0,2,118,0.040442,comp.graphics,rec.sport.hockey
2,0,3,208,0.038561,comp.graphics,sci.space
3,0,4,205,0.034177,comp.graphics,sci.space
4,0,5,79,0.029927,comp.graphics,comp.graphics
...,...,...,...,...,...,...
1495,299,1,191,0.027808,sci.space,rec.sport.hockey
1496,299,2,24,0.026359,sci.space,comp.graphics
1497,299,3,247,0.024057,sci.space,sci.space
1498,299,4,282,0.023873,sci.space,sci.space


## 10. Average Similarity for Same-Class and Different-Class Documents

Every unordered document pair is considered once.

Pairs are divided into:

- **same class:** both documents belong to the same newsgroup
- **different class:** documents belong to different newsgroups

The average cosine similarity is calculated for both groups.

In [14]:
def calculate_average_similarities(similarity_matrix, labels):
    same_class_scores = []
    different_class_scores = []

    n = len(labels)

    for i in range(n):
        for j in range(i + 1, n):
            score = similarity_matrix[i, j]

            if labels[i] == labels[j]:
                same_class_scores.append(score)
            else:
                different_class_scores.append(score)

    return (
        np.mean(same_class_scores),
        np.mean(different_class_scores)
    )

unigram_same, unigram_different = calculate_average_similarities(
    unigram_similarity,
    labels
)

bigram_same, bigram_different = calculate_average_similarities(
    bigram_similarity,
    labels
)

print("TF-IDF Unigrams")
print("----------------")
print(f"Average same-class similarity: "
      f"{unigram_same:.6f}")

print(f"Average different-class similarity: "
      f"{unigram_different:.6f}")

print("\nTF-IDF Unigrams + Bigrams")
print("--------------------------")
print(f"Average same-class similarity: "
      f"{bigram_same:.6f}")

print(f"Average different-class similarity: "
      f"{bigram_different:.6f}")

TF-IDF Unigrams
----------------
Average same-class similarity: 0.025029
Average different-class similarity: 0.011534

TF-IDF Unigrams + Bigrams
--------------------------
Average same-class similarity: 0.012259
Average different-class similarity: 0.005376


## 11. Five Document-Pair Examples

At least five document pairs are required.

Rather than arbitrarily selecting five pairs, this section selects:

- two highly similar same-class pairs
- one highly similar different-class pair
- one moderately similar same-class pair
- one moderately similar different-class pair

This gives a more informative qualitative comparison.

In [15]:
def select_example_pairs(similarity_matrix, labels):
    candidates = []

    n = len(labels)

    for i in range(n):
        for j in range(i + 1, n):
            candidates.append({
                "i": i,
                "j": j,
                "score": similarity_matrix[i, j],
                "same_class": labels[i] == labels[j]
            })

    candidates = pd.DataFrame(candidates)

    same_pairs = candidates[candidates["same_class"]]
    different_pairs = candidates[~candidates["same_class"]]

    # Highest-scoring same-class pairs
    same_high = same_pairs.nlargest(2, "score")

    # Highest-scoring different-class pair
    different_high = different_pairs.nlargest(1, "score")

    # Middle/representative same-class pair
    same_middle = same_pairs.iloc[
        [(len(same_pairs) // 2)]
    ]

    # Middle/representative different-class pair
    different_middle = different_pairs.iloc[
        [(len(different_pairs) // 2)]
    ]

    selected = pd.concat([
        same_high,
        different_high,
        same_middle,
        different_middle
    ]).drop_duplicates(subset=["i", "j"])

    return selected.head(5)

unigram_examples = select_example_pairs(
    unigram_similarity,
    labels
)

bigram_examples = select_example_pairs(
    bigram_similarity,
    labels
)

In [16]:
def format_pair_examples(example_pairs, representation):
    rows = []

    for _, row in example_pairs.iterrows():
        i = int(row["i"])
        j = int(row["j"])

        rows.append({
            "Representation": representation,
            "Document 1": i,
            "Class 1": labels[i],
            "Document 2": j,
            "Class 2": labels[j],
            "Cosine Similarity": row["score"],
            "Same Class": labels[i] == labels[j]
        })

    return pd.DataFrame(rows)

example_table_unigram = format_pair_examples(
    unigram_examples,
    "TF-IDF Unigrams"
)

example_table_bigram = format_pair_examples(
    bigram_examples,
    "TF-IDF Unigrams + Bigrams"
)

pair_examples = pd.concat(
    [example_table_unigram, example_table_bigram],
    ignore_index=True
)

pair_examples["Cosine Similarity"] = (
    pair_examples["Cosine Similarity"].round(6)
)

display(pair_examples)

,Representation,Document 1,Class 1,Document 2,Class 2,Cosine Similarity,Same Class
0,TF-IDF Unigrams,235,sci.space,251,sci.space,1.000000,True
1,TF-IDF Unigrams,233,sci.space,298,sci.space,0.948192,True
2,TF-IDF Unigrams,92,comp.graphics,188,rec.sport.hockey,0.183891,False
3,TF-IDF Unigrams,129,rec.sport.hockey,140,rec.sport.hockey,0.031687,True
4,TF-IDF Unigrams,75,comp.graphics,100,rec.sport.hockey,0.000000,False
5,TF-IDF Unigrams + Bigrams,235,sci.space,251,sci.space,1.000000,True
6,TF-IDF Unigrams + Bigrams,233,sci.space,298,sci.space,0.940615,True
7,TF-IDF Unigrams + Bigrams,68,comp.graphics,134,rec.sport.hockey,0.106297,False
8,TF-IDF Unigrams + Bigrams,129,rec.sport.hockey,140,rec.sport.hockey,0.015469,True
9,TF-IDF Unigrams + Bigrams,75,comp.graphics,100,rec.sport.hockey,0.000000,False


## 12. Final Comparison Table

The final comparison contains only the metrics requested in the question.

The **similarity gap** is included to directly support the discussion of whether adding bigrams improves the ability of TF-IDF to identify same-class documents:

$$
\text{Similarity Gap}
=
\text{Average Same-Class Similarity}
-
\text{Average Different-Class Similarity}
$$

In [17]:
comparison = pd.DataFrame({
    "Representation": [
        "TF-IDF Unigrams",
        "TF-IDF Unigrams + Bigrams"
    ],
    "Number of Documents": [
        X_unigram.shape[0],
        X_bigram.shape[0]
    ],
    "Number of Features": [
        X_unigram.shape[1],
        X_bigram.shape[1]
    ],
    "Average Same-Class Similarity": [
        unigram_same,
        bigram_same
    ],
    "Average Different-Class Similarity": [
        unigram_different,
        bigram_different
    ],
    "Similarity Gap": [
        unigram_same - unigram_different,
        bigram_same - bigram_different
    ]
})

comparison_display = comparison.copy()

for column in [
    "Average Same-Class Similarity",
    "Average Different-Class Similarity",
    "Similarity Gap"
]:
    comparison_display[column] = (
        comparison_display[column].round(6)
    )

display(comparison_display)

,Representation,Number of Documents,Number of Features,Average Same-Class Similarity,Average Different-Class Similarity,Similarity Gap
0,TF-IDF Unigrams,300,6598,0.025029,0.011534,0.013495
1,TF-IDF Unigrams + Bigrams,300,31528,0.012259,0.005376,0.006883


## 13. Comparison and Discussion

The numerical values generated in the previous section should be used to answer the following question:

> Does adding bigrams improve the ability of TF-IDF to identify documents belonging to the same class?

The comparison should focus primarily on the **similarity gap**.

- If the bigram representation has a larger similarity gap, it provides better separation between same-class and different-class documents.
- If the gap is smaller, adding bigrams did not improve class separation.
- If the gap is approximately unchanged, the additional bigram features did not substantially affect the ability to distinguish classes.

A useful interpretation is that unigrams capture individual topic words, while bigrams can capture short phrases and local word associations. However, adding bigrams also increases the number of features and can introduce many sparse, rare features. Therefore, an improvement should be determined from the measured results rather than assumed.

### Automated conclusion

The following cell generates a conclusion based on the actual experimental results.

In [18]:
unigram_gap = unigram_same - unigram_different
bigram_gap = bigram_same - bigram_different

gap_difference = bigram_gap - unigram_gap

print("Unigram similarity gap:")
print(f"{unigram_gap:.6f}")

print("\nUnigram + bigram similarity gap:")
print(f"{bigram_gap:.6f}")

print("\nChange after adding bigrams:")
print(f"{gap_difference:+.6f}")

print("\nConclusion:")

if gap_difference > 0:
    print(
        "Adding bigrams improved the ability of TF-IDF "
        "to distinguish same-class documents from "
        "different-class documents because the similarity "
        "gap increased."
    )
elif gap_difference < 0:
    print(
        "Adding bigrams did not improve the ability of TF-IDF "
        "to distinguish same-class documents from "
        "different-class documents because the similarity "
        "gap decreased."
    )
else:
    print(
        "Adding bigrams produced no change in the similarity "
        "gap, so it did not substantially improve class separation."
    )

Unigram similarity gap:
0.013495

Unigram + bigram similarity gap:
0.006883

Change after adding bigrams:
-0.006613

Conclusion:
Adding bigrams did not improve the ability of TF-IDF to distinguish same-class documents from different-class documents because the similarity gap decreased.


# 14. Final Answer Summary

### Dataset

Three 20 Newsgroups categories were used, with 100 randomly selected documents from each category, resulting in 300 documents.

### Preprocessing

Each document was:

1. converted to lowercase,
2. tokenized,
3. stripped of English stop words,
4. lemmatized.

### TF-IDF representations

Two representations were evaluated:

1. TF-IDF with unigrams.
2. TF-IDF with unigrams and bigrams.

### Cosine similarity

Cosine similarity was implemented manually with NumPy using the dot product and L2 norms. No pre-built cosine similarity function was used.

### Similarity analysis

For every document, its five most similar documents were found and their classes were reported.

The average similarity was calculated separately for:

- documents belonging to the same class
- documents belonging to different classes

### Final comparison

The final table compares both TF-IDF representations using the requested similarity measurements.

The final conclusion is based on whether adding bigrams increases or decreases the difference between same-class and different-class similarity.